In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Input

In [36]:
## Example from CPS 580

# Load the dataset
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, usecols=['Passengers'])

# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(df[['Passengers']])

# Prepare the data for time series prediction
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)
time_step = 10
X, y = create_dataset(data_scaled, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

In [47]:
data_scaled[5:30, 0].shape

(25,)

In [ ]:
# Split data into training and test sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

def build_and_train_lstm():
    model = Sequential()
    model.add(Input(shape=(time_step, 1)))
    model.add(LSTM(50))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)
    return model

# Train LSTM model
print("Training LSTM model...")
lstm_model = build_and_train_lstm()

# Make predictions with LSTM
lstm_predictions = lstm_model.predict(X_test)
lstm_predictions = scaler.inverse_transform(lstm_predictions)
y_test_scaled = scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate RMSE and MAE
rmse = root_mean_squared_error(y_test_scaled, lstm_predictions)
mae = mean_absolute_error(y_test_scaled, lstm_predictions)

print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Error (MAE): {mae}")

In [ ]:
## Lab from CPS 580

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CPS 580/data/tesla_stock_data.csv')
df = df.drop(columns=['Adj Close'])

# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(df[['Close', 'High', 'Low', 'Open', 'Volume']])

# Prepare the data for time series prediction
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)

time_step = 30
X, y = create_dataset(data_scaled, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Split data into training and test sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

def build_and_train_gru():
    model = Sequential()
    model.add(Input(shape=(time_step, 1)))
    model.add(GRU(80))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)
    return model

print("Training GRU model...")
gru_model = build_and_train_gru()

# Make predictions with GRU
gru_predictions = gru_model.predict(X_test)

# Calculate RMSE and MAE
rmse = root_mean_squared_error(y_test, gru_predictions)
mae = mean_absolute_error(y_test, gru_predictions)

print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Error (MAE): {mae}")

In [2]:
PATH = Path('.').absolute()
data_dir = PATH / 'data'

SEED = 1984

In [3]:
df = pd.read_csv(data_dir / 'cleaned_weather.csv')
df['date'] = pd.to_datetime(df['date'])

In [4]:
def create_dataset(df, train_end, valid_end):
    train_df = df.loc[df['date']<=train_end]
    valid_df = df.loc[(train_end < df['date']) & (df['date'] <= valid_end)]
    test_df = df.loc[df['date']>valid_end]

    X_train, y_train = train_df.drop('T', axis=1), train_df['T']
    X_valid, y_valid = valid_df.drop('T', axis=1), valid_df['T']
    X_test, y_test   = test_df.drop('T', axis=1), test_df['T']

    return (
        X_train, X_valid, X_test,
        y_train, y_valid, y_test
    )

In [5]:
train_end = datetime(2020, 6, 30, hour=23, minute=50)
valid_end = datetime(2020, 9, 30, hour=23, minute=50)

(
    X_train, X_valid, X_test, 
    y_train, y_valid, y_test 
) = create_dataset(df, train_end, valid_end)